In [1]:
!pip install requests beautifulsoup4 pandas

In [47]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import os
from urllib.parse import urljoin

# 1. 초기 CSV 세팅
columns = ["cause", "name", "url", "target", "action_name", "action_des", "source", "takes", "time", "need", "level", "saving"]
if not os.path.exists('crawling.csv'):
    pd.DataFrame(columns=columns).to_csv('crawling.csv', index=False, encoding='utf-8-sig')

# 2. 키워드 기반 자동 분류 함수
def guess_category(text):
    if any(k in text for k in ['냉방', '에어컨', '선풍기', '인버터', '압축기']): return '냉방설비'
    if any(k in text for k in ['단열', '창호', '외풍', '리모델링', '변압기']): return '단열 불량'
    if any(k in text for k in ['보일러', '난방', '연탄']): return '난방설비'
    if any(k in text for k in ['태양광', '신재생', '친환경', '햇빛']): return '친환경 설비'
    if any(k in text for k in ['온실가스', '감축', '탄소']): return '절약시설'
    return '운영 습관'

def guess_level(need_text):
    if any(k in need_text for k in ['어려움', '사업자', '확인서', '증명서', '공사']): return '어려움'
    if any(k in need_text for k in ['신청서', '주민등록', '보조금', '제출', '공문']): return '중간'
    if need_text == "상세내용확인필요" or not need_text: return "확인필요"
    return '쉬움'

# 3. 메인 크롤링 함수
def update_rag_data_from_web():
    base_url = "https://www.e-policy.or.kr/web/lay1/program/S1T9C14/curation/list.do"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    response = requests.get(base_url, headers=headers)
    
    if response.status_code != 200:
        print(f"상태 코드 오류: {response.status_code}")
        return

    soup = BeautifulSoup(response.text, 'html.parser')
    cards = soup.select('a.curating__con')
    data = []
    
    print(f"총 {len(cards)}개의 정책 목록을 발견했습니다. 상세 데이터 추출을 시작합니다...\n")
    
    for idx, card in enumerate(cards):
        # 1차 정보: 리스트 페이지 추출
        raw_text = card.select_one('p.tit').text.strip()
        clean_title = re.sub(r'\[D-\d+\]|\[마감\]', '', raw_text).strip()
        
        target = "확인필요"
        takes = "확인필요"
        apply_time = "확인필요"
        
        wraps = card.select('.txt-wrap')
        for wrap in wraps:
            tit_tag = wrap.select_one('p.tit')
            txt_tag = wrap.select_one('p.txt')
            
            if tit_tag and txt_tag:
                tit_val = tit_tag.text.strip()
                txt_val = txt_tag.text.strip()
                
                if "지원대상" in tit_val:
                    target = txt_val
                elif "지원혜택" in tit_val:
                    takes = txt_val
                elif "신청기간" in tit_val:
                    apply_time = txt_val
        
        # URL 추출 
        link = urljoin(base_url, card.get('href'))
        
        # 2차 정보: 기본값 세팅
        action_des = "상세내용확인필요"
        need = "상세내용확인필요"
        source = "한국에너지정보문화재단"
        final_url = link 
        
        print(f"[{idx+1}/{len(cards)}] 수집 중: {clean_title}")
        
        try:
            detail_res = requests.get(link, headers=headers)
            if detail_res.status_code == 200:
                detail_soup = BeautifulSoup(detail_res.text, 'html.parser')
                
                # 사이트 바로가기 버튼 및 URL 매핑
                site_btn = detail_soup.find(lambda tag: tag.name == 'a' and ('사이트 바로가기' in tag.get_text() or '홈페이지' in tag.get_text()))
                if site_btn and site_btn.get('href'):
                    source_url = site_btn.get('href')
                    final_url = source_url 
                    
                    domain_dict = {
                        "mois.go.kr": "행정안전부", "motie.go.kr": "산업통상자원부", 
                        "energy.or.kr": "한국에너지공단", "kepco.co.kr": "한국전력공사",
                        "seoul.go.kr": "서울특별시", "gg.go.kr": "경기도",
                        "https://www.nowon.kr/www/user/bbs/BD_selectBbs.do?q_bbsCode=1003&q_bbscttSn=20260406114143655&q_estnColumn7=&q_estnColumn1=11&q_ntceSiteCode=11&q_clCode=0&q_rowPerPage=10&q_currPage=1&q_sortName=&q_sortOrder=&q_searchKeyTy=sj___1002&q_searchVal=&":"노원구청",
                        "https://www.gg.go.kr/bbs/boardView.do?bIdx=230465845&bsIdx=469&bcIdx=0&menuId=1547&isManager=false&isCharge=false&keyfield=SUBJECTANDREMARK&keyword=%EC%97%90%EB%84%88%EC%A7%80&page=1":"경기도청",
                        "https://www.bokjiro.go.kr/":"보건복지부, 복지로",
                        "https://www.energy.or.kr/":"한국에너지공단",
                        "https://koat2026.careeron.co.kr/#/":"한국농업기술진흥원",
                        "https://online.kepco.co.kr/":"한국전력공사"
                    }
                    matched_source = None
                    for domain, org_name in domain_dict.items():
                        if domain in source_url:
                            matched_source = org_name
                            break
                    source = matched_source if matched_source else source_url
                
                content_area = detail_soup.select_one('.content')
                if content_area:
                    full_text = content_area.get_text(separator=" ", strip=True)
                    
                    # 지원내용
                    if '지원내용' in full_text:
                        chunk = full_text.split('지원내용')[1] 
                        for cutoff in ['관련정보', '신청방법', '신청기간', '목록으로']:
                            if cutoff in chunk:
                                chunk = chunk.split(cutoff)[0]
                        action_des = chunk.strip()
                        
                    # 신청방법
                    if '신청방법' in full_text:
                        chunk = full_text.split('신청방법')[1] 
                        for cutoff in ['문의처', '유의사항', '목록으로', '사이트 바로가기']:
                            if cutoff in chunk:
                                chunk = chunk.split(cutoff)[0]
                        need = chunk.strip()
                        
        except Exception as e:
            print(f"  -> 오류 발생: {e}")
            
        time.sleep(1) # 대기
        
        # 3차 정보: 가공 및 요약
        search_context = clean_title + " " + action_des
        cause_val = guess_category(search_context)
        level_val = guess_level(need)
        action_name = clean_title[:15] + ("..." if len(clean_title) > 15 else "")
        
        data.append({
            "cause": cause_val,
            "name": clean_title,
            "url": final_url,
            "target": target,
            "action_name": action_name,
            "action_des": action_des,
            "source": source,
            "takes": takes,
            "time": apply_time,
            "need": need,
            "level": level_val,
            "saving": "기존 설비 대비 절감 (추정)"
        })
        
    # 결과 저장
    new_df = pd.DataFrame(data)
    new_df = new_df.replace('\xa0', ' ', regex=True) 

    existing_df = pd.read_csv('crawling.csv', encoding='utf-8-sig')
    updated_df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(['name'])
    updated_df.to_csv('crawling.csv', index=False, encoding='utf-8-sig')
    print(f"업데이트 완료")
    display(updated_df.tail(5))

# 실행
update_rag_data_from_web()

총 9개의 정책 목록을 발견했습니다. 상세 데이터 추출을 시작합니다...

[1/9] 수집 중: 2026 햇빛소득마을 선정 공고
[2/9] 수집 중: 2026 서울 동작형 긴급 에너지 지원
[3/9] 수집 중: 2026 서울 노원 베란다형 미니태양광 보급사업
[4/9] 수집 중: 2026 경기도 에너지 융자지원
[5/9] 수집 중: 6월 15일부터 에너지 바우처 신청하세요!
[6/9] 수집 중: 중소·중견기업 온실가스 감축 인프라 구축 지원
[7/9] 수집 중: 농업농촌 자발적 온실가스 감축사업
[8/9] 수집 중: 고효율 터보압축기
[9/9] 수집 중: 고효율 변압기
업데이트 완료


,cause,name,url,target,action_name,action_des,source,takes,time,need,level,saving
4,운영 습관,6월 15일부터 에너지 바우처 신청하세요!,https://www.bokjiro.go.kr/,"영유아,아동,청소년,청년,중장년,노년기",6월 15일부터 에너지 바우...,"○ 지원 금액 - 1인 세대: 295,200원 - 2인 세대: 407,500원 - ...","보건복지부, 복지로","냉·난방비(전기, 도시가스, 지역난방, 등유, LPG, 연탄 등)",2026.06.15 ~ 2026.12.31,○ 읍·면·동 행정복지센터 방문 ○ 온라인: 복지로 홈페이지(www.bokjiro....,쉬움,기존 설비 대비 절감 (추정)
5,절약시설,중소·중견기업 온실가스 감축 인프라 구축 지원,https://www.energy.or.kr/,"노년기,중장년,청년",중소·중견기업 온실가스 감축...,○ 기업규모에 따라 차등지원 - 중견기업: 총 투자금의 40% - 중소기업: 총 투...,한국에너지공단,현금,2024.01.01 ~ 2026.12.31,○ 방문장소 및,쉬움,기존 설비 대비 절감 (추정)
6,절약시설,농업농촌 자발적 온실가스 감축사업,https://koat2026.careeron.co.kr/#/,"노년기,중장년,청년,청소년",농업농촌 자발적 온실가스 감...,"○ 지원형태: 국고보조 100% ○ 내용 - 컨설팅 지원(4종류: 사업계획서, 모니...",한국농업기술진흥원,​​​​​‌​​​‌​​‌​​​​‌​​‌‌‌​​​​‌‌‌‌‌​‌​​​​‌‌100% 보조,2025.01.01 ~ 2026.12.31,"○ 우편, 팩스: 한국농업기술진흥원",쉬움,기존 설비 대비 절감 (추정)
7,냉방설비,고효율 터보압축기,https://online.kepco.co.kr/,"노년기,중장년,청년,청소년",고효율 터보압축기,○ 에너지 효율향상 및 온실가스 감축을 위해 고객이 고효율 기기 교체 시 지원금 지급,한국전력공사,​​​​​‌​​​‌​​‌​​​​‌​​‌‌‌​​​​‌‌‌‌‌​‌​​​​‌‌교체 지원금,2024.01.01 ~ 2026.12.31,○ 온라인: 한전 사이버지점(http://cyber.kepco.co.kr/) ○ 방...,쉬움,기존 설비 대비 절감 (추정)
8,단열 불량,고효율 변압기,https://online.kepco.co.kr/,"노년기,중장년,청소년,청년",고효율 변압기,○ 에너지 효율향상 및 온실가스 감축을 위해 고객이 고효율 기기 교체 시 지원금 지급,한국전력공사,​​​​​‌​​​‌​​‌​​​​‌​​‌‌‌​​​​‌‌‌‌‌​‌​​​​‌‌고효율 변압...,2024.01.01 ~ 2026.12.31,○ 온라인: 한전 사이버지점(http://cyber.kepco.co.kr/) ○ 방...,쉬움,기존 설비 대비 절감 (추정)
